In [0]:
folder_path = '/Volumes/ai_capabilities/document_intelligence/invoices/load_invoice_1/test_invoices/test_invoices/'

In [0]:
from pyspark.sql import functions as F
new_df = (spark.read.format("binaryFile").load(f"{folder_path}doc*.pdf")
    .select("path", F.expr("ai_parse_document(content, map('version','2.0')) AS parsed")))
new_df.write.mode("append").saveAsTable("ai_capabilities.document_intelligence.bronze_invoices_parsed")

In [0]:
%sql
select * from ai_capabilities.document_intelligence.bronze_invoices_parsed

In [0]:
df = spark.read.table("ai_capabilities.document_intelligence.bronze_invoices_parsed")
labels = ('{"invoice":"Seller invoice requesting payment for goods or services",'
          '"credit_note":"Document that reduces or cancels an amount owed, with negative amounts",'
          '"delivery_note":"Lists delivered goods, no payment request",'
          '"purchase_order":"Buyer order sent to a supplier",'
          '"other":"Any other document"}')
df.select("path",
    F.element_at(F.split("path", "/"), -1).alias("file_name"),
    F.expr("ai_classify(parsed, '" + labels + "', map('version','2.1','enableConfidenceScores','true','instructions','Business documents received by accounts payable.'))").alias("cls")
).write.format("delta").mode("overwrite").saveAsTable("ai_capabilities.document_intelligence.silver_doc_class")

In [0]:
from pyspark.sql import functions as F
class_df = spark.read.table("ai_capabilities.document_intelligence.silver_doc_class")
(class_df.select(
    "path",
    "file_name",
    F.expr("cls:response[0].value::string").alias("doc_type"),
    F.expr("cls:response[0].confidence_score::double").alias("conf"),
    F.expr("""CASE
        WHEN cls:response[0].value::string = 'invoice' AND cls:response[0].confidence_score::double >= 0.8 THEN 'EXTRACT'
        WHEN cls:response[0].value::string = 'invoice' THEN 'REVIEW_LOW_CONF'
        ELSE 'OTHER_DOC'
    END""").alias("route")
).write.mode("overwrite").saveAsTable("ai_capabilities.document_intelligence.doc_routing"))


In [0]:
%sql
select * from ai_capabilities.document_intelligence.doc_routing

In [0]:
%skip
%sql
-- CREATE OR REPLACE TABLE ai_capabilities.document_intelligence.doc_routing AS
-- SELECT path, file_name,
--   cls:response[0].value::string AS doc_type,
--   cls:response[0].confidence_score::double AS conf,
--   CASE WHEN cls:response[0].value::string = 'invoice' AND cls:response[0].confidence_score::double >= 0.8 THEN 'EXTRACT'
--        WHEN cls:response[0].value::string = 'invoice' THEN 'REVIEW_LOW_CONF'
--        ELSE 'OTHER_DOC' END AS route
-- FROM ai_capabilities.document_intelligence.silver_doc_class;

In [0]:
%skip
%sql
-- CREATE OR REPLACE TABLE ai_capabilities.document_intelligence.silver_doc_class AS
-- SELECT path, element_at(split(path,'/'),-1) AS file_name,
--   ai_classify(parsed,
--     '{"invoice":"Seller invoice requesting payment for goods or services",
--       "credit_note":"Document that reduces or cancels an amount owed, with negative amounts",
--       "delivery_note":"Lists delivered goods, no payment request",
--       "purchase_order":"Buyer order sent to a supplier",
--       "other":"Any other document"}',
--     map('version','2.1','enableConfidenceScores','true',
--         'instructions','Business documents received by accounts payable.')) AS cls
-- FROM ai_capabilities.document_intelligence.bronze_invoices_parsed;

SELECT file_name, cls FROM ai_capabilities.document_intelligence.silver_doc_class;

# Pipeline overview

Classification & routing stage of the IDP pipeline: ingest the non-invoice documents, classify everything in bronze, and assign each document a route.

* **Cell 1 — Source path setup.** Defines `folder_path` pointing at the test-invoice volume; feeds the binary read in Cell 2.
* **Cell 2 — Parse non-invoice docs.** Reads `doc*.pdf` (credit note, delivery note, purchase order) and appends them — parsed with `ai_parse_document` — to the shared bronze table `bronze_invoices_parsed`. Extends the same bronze layer the sibling notebooks build.
* **Cell 3 — Bronze peek.** SQL `SELECT *` on `bronze_invoices_parsed` — a checkpoint of what has landed before classification.
* **Cell 4 — Classify all documents.** Reads the full bronze table and classifies every parsed document with `ai_classify` (5 labels including `other`, each with a description, plus confidence scores), overwriting `silver_doc_class`. Its output drives routing in Cell 5. Note the overwrite (not incremental) — re-running it reclassifies every bronze document.
* **Cell 5 — Build routing table.** Reads `silver_doc_class`, extracts `doc_type` and confidence, and assigns a route — `EXTRACT` (invoice with conf ≥ 0.8), `REVIEW_LOW_CONF` (invoice below threshold), `OTHER_DOC` (everything else) — overwriting `doc_routing`. This is the routing table downstream extraction steps consume.
* **Cell 6 — Routing peek.** SQL `SELECT *` on `doc_routing` — verification checkpoint of the routing decisions.

**Overall flow:** Cell 1 (path) → Cell 2 (parse doc*.pdf into bronze) → Cell 3 (peek) → Cell 4 (classify → `silver_doc_class`) → Cell 5 (route → `doc_routing`) → Cell 6 (peek), with Cells 7–8 as SQL-variant alternates of those same two steps.

**Downstream:** the Image processing notebook consumes `silver_doc_class` for classification-aware extraction; `doc_routing` remains the routing feed for consumers that follow the confidence-threshold logic.